# Tutorial: generating the controlled CLEVR datasets

Two datasets for analysing LLaVA-1.5, with objects aligned to its 14 px patch grid:

- `CLEVR_SINGLE_OBJECT`: one object per image, 36 positions × 3 shapes × 8 colors = 864 images
- `CLEVR_DUAL_OBJECT`: two objects per image, 9 positions, 3 shapes, red/blue = 1,296 images

**How I made this:** I did it with Claude. I told it I want one object per image, on a grid that lines up with LLaVA's 14 px patches, every object the same size wherever it is, 3 shapes and 8 colors, and then a second dataset with two objects that never share a patch. It wrote the scripts and the `settings.json` files and automated the whole thing for me, went really fast.

Each dataset is one folder in `output/`, and everything that defines it is already in there:

| Folder | Files |
|---|---|
| `output/CLEVR_SINGLE_OBJECT/` | `settings.json` (what the dataset should look like), `placements.json` (camera and where each object goes, measured in Blender) |
| `output/CLEVR_DUAL_OBJECT/` | `settings.json`, `grid_placements.json` (the single-object placements it builds on), `layout.json` (which positions and position pairs are used) |

You need Blender 5.x and Python 3. Run this notebook from the repository root.

In [ ]:
BLENDER = '/Applications/Blender.app/Contents/MacOS/Blender'   # change if Blender is elsewhere

## 1. Replicate the existing datasets

Nothing to set up, just render. Single-object dataset (about 25 min), exported in the CLEVR format to `images/train/` and `scenes/CLEVR_train_scenes.json`:

In [ ]:
!cd image_generation && {BLENDER} --background --python-exit-code 1 --python render_single_object_dataset.py -- --config ../output/CLEVR_SINGLE_OBJECT/settings.json --stage render 2>&1 | grep -v "Saved:"
!cd image_generation && python3 export_clevr_format.py --dataset single --source_dir ../output/CLEVR_SINGLE_OBJECT/generation --target_dir ../output/CLEVR_SINGLE_OBJECT --overwrite 1

Two-object dataset, with a check (`verify`) and a contact sheet (`preview`):

In [ ]:
!cd image_generation && {BLENDER} --background --python-exit-code 1 --python render_two_object_dataset.py -- --config ../output/CLEVR_DUAL_OBJECT/settings.json --stage render verify preview 2>&1 | grep -v "Saved:"
!cd image_generation && python3 export_clevr_format.py --dataset dual --source_dir ../output/CLEVR_DUAL_OBJECT/generation --target_dir ../output/CLEVR_DUAL_OBJECT --overwrite 1

## 2. A new setup

Make a new folder in `output/` with a `settings.json` in it. Easiest is to ask Claude, e.g. *"make me a settings.json for a single-object dataset with a 4×4 grid and no shadows"*. The two existing `settings.json` files show every option; anything you leave out stays as in the existing datasets.

For example, `output/MY_SINGLE_OBJECT/settings.json`:

```json
{
  "layout": {"grid_size": 4, "grid_spacing_patches": 4},
  "render": {"cast_shadows": 0},
  "paths": {"placements_json": "placements.json", "output_dir": "generation"}
}
```

Then run it. This first computes `placements.json` (about 2 minutes), then renders the images:

In [ ]:
NEW = 'output/MY_SINGLE_OBJECT'
!cd image_generation && {BLENDER} --background --python-exit-code 1 --python render_single_object_dataset.py -- --config ../{NEW}/settings.json --stage all 2>&1 | grep -v "Saved:"
!cd image_generation && python3 export_clevr_format.py --dataset single --source_dir ../{NEW}/generation --target_dir ../{NEW} --overwrite 1

A two-object dataset works the same way, its `settings.json` just points to the single-object `placements.json` it builds on. For example, `output/MY_DUAL_OBJECT/settings.json`:

```json
{
  "layout": {"grid_size": 2, "grid_step": 2},
  "render": {"cast_shadows": 0},
  "paths": {"single_placements_json": "../MY_SINGLE_OBJECT/placements.json", "layout_json": "layout.json", "output_dir": "generation"}
}
```

In [ ]:
NEW_DUAL = 'output/MY_DUAL_OBJECT'
!cd image_generation && {BLENDER} --background --python-exit-code 1 --python render_two_object_dataset.py -- --config ../{NEW_DUAL}/settings.json --stage layout render verify preview 2>&1 | grep -v "Saved:"
!cd image_generation && python3 export_clevr_format.py --dataset dual --source_dir ../{NEW_DUAL}/generation --target_dir ../{NEW_DUAL} --overwrite 1

## What you get

```
output/<dataset>/
├── settings.json, placements.json (+ grid_placements.json, layout.json for two objects)
├── images/train/CLEVR_train_000000.png, ...   the images (CLEVR format)
├── scenes/CLEVR_train_scenes.json             the scene annotations (CLEVR format)
└── generation/                                masks, patch coverage, metadata, logs
                                               (two objects: questions, matched-pair groupings, verification, contact sheet)
```

Only the files in the first line are tracked in Git, the rest is rendered. `visualize_patch_grid.ipynb` shows how the objects fall into the patch grid. The design is explained in `image_generation/data/vlm_single_object/README.md`.